# 🌿 PlantScan — Optimized Training Notebook
### Run this on Google Colab with GPU enabled

**Before starting:**
1. Click `Runtime` → `Change runtime type` → Choose **T4 GPU** → Save
2. Run each cell from top to bottom
3. At the end, download the zip file containing your model

---
**What this fixes vs your original notebook:**
- ❌ Original used `rescale=1./255` on EfficientNet → WRONG, kills accuracy
- ❌ Original trained only 7 epochs — far too few
- ❌ Original froze wrong layers during fine-tuning
- ✅ This uses correct EfficientNetV2B2 built-in preprocessing
- ✅ Two-phase training: feature extraction then fine-tuning
- ✅ Cosine LR decay, label smoothing, class weights, TTA

In [ ]:
# ═══════════════════════════════════════════════
# CELL 1 — Check GPU
# ═══════════════════════════════════════════════
import tensorflow as tf
print('TensorFlow:', tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print('GPU available:', gpus)
if not gpus:
    print('⚠️  No GPU found! Go to Runtime → Change runtime type → T4 GPU')
else:
    print('✅ GPU is ready!')

In [ ]:
# ═══════════════════════════════════════════════
# CELL 2 — Install kaggle for dataset download
# ═══════════════════════════════════════════════
!pip install -q kaggle
print('Done')

In [ ]:
# ═══════════════════════════════════════════════
# CELL 3 — Upload kaggle.json and download dataset
# ═══════════════════════════════════════════════
# HOW TO GET kaggle.json:
#   1. Go to https://www.kaggle.com
#   2. Click your profile → Settings → API → Create New Token
#   3. A file called kaggle.json will download
#   4. Upload it when prompted below

from google.colab import files
import os

print('Upload your kaggle.json file:')
uploaded = files.upload()  # ← choose kaggle.json here

os.makedirs('/root/.config/kaggle', exist_ok=True)
!cp kaggle.json /root/.config/kaggle/
!chmod 600 /root/.config/kaggle/kaggle.json

print('Downloading PlantVillage dataset (~1.5 GB)...')
!kaggle datasets download -d abdallahalidev/plantvillage-dataset
!unzip -q plantvillage-dataset.zip -d /content/plantvillage_data
print('Done!')
!ls /content/plantvillage_data/

In [ ]:
# ═══════════════════════════════════════════════
# CELL 4 — Locate the data directory automatically
# ═══════════════════════════════════════════════
import os

DATA_DIR = None
for root, dirs, files_list in os.walk('/content/plantvillage_data'):
    # The correct folder has many class subdirectories
    subdirs = [d for d in dirs if os.path.isdir(os.path.join(root, d))]
    if len(subdirs) >= 10:
        DATA_DIR = root
        break

if DATA_DIR is None:
    # Fallback — try common paths
    for path in [
        '/content/plantvillage_data/plantvillage dataset/color',
        '/content/plantvillage_data/color',
        '/content/plantvillage_data',
    ]:
        if os.path.exists(path):
            DATA_DIR = path
            break

print(f'Data directory: {DATA_DIR}')
classes = [d for d in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, d))]
print(f'Number of classes: {len(classes)}')
print(f'Sample classes: {classes[:5]}')

In [ ]:
# ═══════════════════════════════════════════════
# CELL 5 — All imports and config
# ═══════════════════════════════════════════════
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, mixed_precision
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120
import numpy as np
import json
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
from pathlib import Path
from collections import Counter

# Enable mixed precision — doubles training speed on T4/A100
mixed_precision.set_global_policy('mixed_float16')
print('Mixed precision:', mixed_precision.global_policy().name)

# ── Config ─────────────────────────────────────────
MODELS_DIR    = '/content/models'
IMG_SIZE      = (260, 260)   # EfficientNetV2B2 native resolution
BATCH_SIZE    = 32
SEED          = 42
AUTOTUNE      = tf.data.AUTOTUNE
PHASE1_EPOCHS = 15           # feature extraction
PHASE2_EPOCHS = 25           # fine-tuning
PHASE1_LR     = 1e-3
PHASE2_LR     = 2e-5

Path(MODELS_DIR).mkdir(parents=True, exist_ok=True)

NUM_CLASSES = len([d for d in os.listdir(DATA_DIR)
                   if os.path.isdir(os.path.join(DATA_DIR, d))])
print(f'✅ {NUM_CLASSES} classes found')

In [ ]:
# ═══════════════════════════════════════════════
# CELL 6 — Build tf.data pipeline
# ═══════════════════════════════════════════════
# CRITICAL FIX: EfficientNetV2 handles its own normalisation.
# Feed raw [0, 255] pixels — do NOT rescale to [0, 1].
# The original notebook's rescale=1./255 was the main cause of 44% accuracy.

train_ds_raw = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR, validation_split=0.2, subset='training',
    seed=SEED, image_size=IMG_SIZE,
    batch_size=BATCH_SIZE, label_mode='categorical',
)
val_ds_raw = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR, validation_split=0.2, subset='validation',
    seed=SEED, image_size=IMG_SIZE,
    batch_size=BATCH_SIZE, label_mode='categorical',
    shuffle=False,
)

CLASS_NAMES = train_ds_raw.class_names
print(f'Sample class names: {CLASS_NAMES[:5]}')

# Save class index map now (needed by the Streamlit app)
class_index = {name: i for i, name in enumerate(CLASS_NAMES)}
with open(f'{MODELS_DIR}/class_indices.json', 'w') as f:
    json.dump(class_index, f, indent=2)

# Augmentation applied during training only
augment = keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.2),
    layers.RandomZoom(0.2),
    layers.RandomTranslation(0.1, 0.1),
    layers.RandomBrightness(0.2),
    layers.RandomContrast(0.2),
], name='augmentation')

train_ds = (train_ds_raw
            .map(lambda x, y: (augment(x, training=True), y),
                 num_parallel_calls=AUTOTUNE)
            .prefetch(AUTOTUNE))
val_ds = val_ds_raw.prefetch(AUTOTUNE)

print(f'Train batches: {len(train_ds_raw)} | Val batches: {len(val_ds_raw)}')

In [ ]:
# ═══════════════════════════════════════════════
# CELL 7 — Compute class weights for imbalanced data
# ═══════════════════════════════════════════════
label_counts = Counter()
for _, labels in train_ds_raw.unbatch():
    label_counts[int(tf.argmax(labels).numpy())] += 1

total = sum(label_counts.values())
class_weights = {cls: total / (NUM_CLASSES * cnt) for cls, cnt in label_counts.items()}

print(f'Total training images: {total:,}')
print(f'Class weight range: {min(class_weights.values()):.3f} – {max(class_weights.values()):.3f}')

In [ ]:
# ═══════════════════════════════════════════════
# CELL 8 — Build model
# ═══════════════════════════════════════════════
def build_model(num_classes, img_size):
    inputs = keras.Input(shape=(*img_size, 3), name='input')

    # EfficientNetV2B2 with include_preprocessing=True:
    # it applies its own normalisation internally — no manual rescale needed
    backbone = tf.keras.applications.EfficientNetV2B2(
        include_top=False,
        weights='imagenet',
        input_tensor=inputs,
        include_preprocessing=True,
    )
    backbone.trainable = False  # Phase 1: frozen

    x = backbone.output
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dense(512)(x)
    x = layers.Activation('swish')(x)   # swish works better than relu with EfficientNet
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(256)(x)
    x = layers.Activation('swish')(x)
    x = layers.Dropout(0.3)(x)
    # float32 output for numerical stability with mixed precision
    outputs = layers.Dense(num_classes, activation='softmax', dtype='float32')(x)

    model = keras.Model(inputs, outputs)
    return model, backbone

model, backbone = build_model(NUM_CLASSES, IMG_SIZE)
trainable = sum(p.numpy().size for p in model.trainable_variables)
total_p   = sum(p.numpy().size for p in model.variables)
print(f'Trainable: {trainable:,} / {total_p:,} params')

In [ ]:
# ═══════════════════════════════════════════════
# CELL 9 — Phase 1: Feature extraction
# ═══════════════════════════════════════════════
lr_p1 = keras.optimizers.schedules.CosineDecayRestarts(
    initial_learning_rate=PHASE1_LR,
    first_decay_steps=len(train_ds_raw) * 5,
    t_mul=1.5, m_mul=0.9, alpha=1e-5,
)
model.compile(
    optimizer=keras.optimizers.Adam(lr_p1),
    loss=keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=['accuracy', keras.metrics.TopKCategoricalAccuracy(k=3, name='top3')],
)

cb_p1 = [
    EarlyStopping(patience=6, restore_best_weights=True,
                  monitor='val_accuracy', mode='max', verbose=1),
    ModelCheckpoint(f'{MODELS_DIR}/phase1_best.keras',
                    save_best_only=True, monitor='val_accuracy', mode='max'),
]

print('=' * 55)
print('PHASE 1 — Training classification head (backbone frozen)')
print('=' * 55)
history_p1 = model.fit(
    train_ds, validation_data=val_ds,
    epochs=PHASE1_EPOCHS, callbacks=cb_p1,
    class_weight=class_weights, verbose=1,
)

_, acc_p1, top3_p1 = model.evaluate(val_ds, verbose=0)
print(f'\n📊 Phase 1 → Accuracy: {acc_p1*100:.2f}% | Top-3: {top3_p1*100:.2f}%')

In [ ]:
# ═══════════════════════════════════════════════
# CELL 10 — Phase 2: Fine-tuning
# ═══════════════════════════════════════════════
backbone.trainable = True
freeze_until = int(len(backbone.layers) * 0.60)
for i, layer in enumerate(backbone.layers):
    # freeze bottom 60% + always keep BatchNorm frozen
    if i < freeze_until or isinstance(layer, layers.BatchNormalization):
        layer.trainable = False
    else:
        layer.trainable = True

trainable_now = sum(1 for l in model.layers if l.trainable)
print(f'Trainable layers: {trainable_now} / {len(model.layers)}')

lr_p2 = keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=PHASE2_LR,
    decay_steps=len(train_ds_raw) * PHASE2_EPOCHS,
    alpha=1e-7,
)
model.compile(
    optimizer=keras.optimizers.Adam(lr_p2),
    loss=keras.losses.CategoricalCrossentropy(label_smoothing=0.05),
    metrics=['accuracy', keras.metrics.TopKCategoricalAccuracy(k=3, name='top3')],
)

cb_p2 = [
    EarlyStopping(patience=8, restore_best_weights=True,
                  monitor='val_accuracy', mode='max', verbose=1),
    ModelCheckpoint(f'{MODELS_DIR}/plant_disease_model_final.keras',
                    save_best_only=True, monitor='val_accuracy',
                    mode='max', verbose=1),
]

print('=' * 55)
print('PHASE 2 — Fine-tuning top 40% of backbone')
print('=' * 55)
history_p2 = model.fit(
    train_ds, validation_data=val_ds,
    epochs=PHASE2_EPOCHS, callbacks=cb_p2,
    class_weight=class_weights, verbose=1,
)

_, acc_p2, top3_p2 = model.evaluate(val_ds, verbose=0)
print(f'\n📊 Phase 2 → Accuracy: {acc_p2*100:.2f}% | Top-3: {top3_p2*100:.2f}%')

In [ ]:
# ═══════════════════════════════════════════════
# CELL 11 — TTA Evaluation
# ═══════════════════════════════════════════════
tta_aug = keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])

all_preds, all_labels = [], []
for images, labels in val_ds:
    batch_preds = tf.zeros((tf.shape(images)[0], NUM_CLASSES))
    for _ in range(10):
        batch_preds = batch_preds + model(tta_aug(images, training=True), training=False)
    all_preds.append(batch_preds / 10)
    all_labels.append(labels)

preds_arr  = tf.concat(all_preds, 0).numpy()
labels_arr = tf.concat(all_labels, 0).numpy()
pred_cls   = np.argmax(preds_arr, 1)
true_cls   = np.argmax(labels_arr, 1)
tta_acc    = np.mean(pred_cls == true_cls)
tta_top3   = np.mean([true_cls[i] in np.argsort(preds_arr[i])[-3:] for i in range(len(true_cls))])

print(f'\n🎯 FINAL RESULTS (with TTA x10):')
print(f'   Accuracy : {tta_acc*100:.2f}%')
print(f'   Top-3    : {tta_top3*100:.2f}%')
print(f'\n📈 Improvement: baseline 44% → {tta_acc*100:.2f}%')

In [ ]:
# ═══════════════════════════════════════════════
# CELL 12 — Training curves
# ═══════════════════════════════════════════════
acc      = history_p1.history['accuracy']     + history_p2.history['accuracy']
val_acc  = history_p1.history['val_accuracy'] + history_p2.history['val_accuracy']
loss     = history_p1.history['loss']         + history_p2.history['loss']
val_loss = history_p1.history['val_loss']     + history_p2.history['val_loss']
split    = len(history_p1.history['accuracy'])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Training History', fontsize=14, fontweight='bold')

for ax, tr, va, title in [
    (ax1, acc, val_acc, 'Accuracy'),
    (ax2, loss, val_loss, 'Loss'),
]:
    ax.plot(tr, label='Train', color='#4ade80')
    ax.plot(va, label='Validation', color='#60a5fa')
    ax.axvline(split - 1, color='red', linestyle='--', alpha=0.7, label='Fine-tune start')
    ax.set_title(title)
    ax.set_xlabel('Epoch')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{MODELS_DIR}/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved training_curves.png')

In [ ]:
# ═══════════════════════════════════════════════
# CELL 13 — Confusion matrix (top 15 classes)
# ═══════════════════════════════════════════════
cm = confusion_matrix(true_cls, pred_cls)

# Show only top 15 most confused classes for readability
errors_per_class = cm.sum(axis=1) - np.diag(cm)
top15 = np.argsort(errors_per_class)[-15:][::-1]
cm_sub = cm[np.ix_(top15, top15)]
labels_sub = [CLASS_NAMES[i][:20] for i in top15]

plt.figure(figsize=(14, 12))
sns.heatmap(cm_sub, annot=True, fmt='d', cmap='Greens',
            xticklabels=labels_sub, yticklabels=labels_sub)
plt.title('Confusion Matrix — Top 15 Most Confused Classes', fontsize=13, fontweight='bold')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout()
plt.savefig(f'{MODELS_DIR}/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ═══════════════════════════════════════════════
# CELL 14 — Save model + metadata
# ═══════════════════════════════════════════════
# Save in native Keras format (better than .h5)
model.save(f'{MODELS_DIR}/plant_disease_model_final.keras')

# Save metadata — the Streamlit app reads this
metadata = {
    'num_classes'   : NUM_CLASSES,
    'class_names'   : CLASS_NAMES,
    'img_size'      : list(IMG_SIZE),
    'accuracy_p1'   : float(acc_p1),
    'accuracy_p2'   : float(acc_p2),
    'accuracy_tta'  : float(tta_acc),
    'top3_tta'      : float(tta_top3),
    'backbone'      : 'EfficientNetV2B2',
    'preprocessing' : 'built-in (include_preprocessing=True)',
    'note'          : 'Feed raw [0,255] images. Do NOT rescale.',
}
with open(f'{MODELS_DIR}/model_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print('Files in /content/models/:')
for fname in sorted(os.listdir(MODELS_DIR)):
    size_mb = os.path.getsize(f'{MODELS_DIR}/{fname}') / 1e6
    print(f'  {fname:45s}  {size_mb:7.1f} MB')

In [ ]:
# ═══════════════════════════════════════════════
# CELL 15 — Download everything
# ═══════════════════════════════════════════════
from google.colab import files

!zip -r /content/plantscan_models.zip /content/models/
files.download('/content/plantscan_models.zip')

print('\n✅ Download started!')
print('\nNext steps:')
print('  1. Unzip plantscan_models.zip')
print('  2. Copy all files into the plantscan/models/ folder')
print('  3. Run: streamlit run src/app.py')

In [ ]:
# ═══════════════════════════════════════════════
# CELL 16 — Classification report
# ═══════════════════════════════════════════════
print(classification_report(true_cls, pred_cls,
                             target_names=CLASS_NAMES, digits=3))